# Shared, executable implementation for Part 1
All five model notebooks execute these definition cells. Data preparation, experiments and outputs remain in notebooks. No Python module is required. The assignment prohibits LLM assistance with the submitted writeup; markdown here documents software and measurement protocols, not a submission report.

Sources: [assignment](../../ANLP-Assignment-2.pdf), [dataset](https://huggingface.co/datasets/belumind/en-vi-ja-curated-500k-triplets), [Switch Transformer](https://arxiv.org/abs/2101.03961), [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html), [SacreBLEU](https://github.com/mjpost/sacrebleu).


In [ ]:
import os, json, math, time, random, hashlib, contextlib, platform, unicodedata, gc, io
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, decoders, trainers
import sacrebleu
from IPython.display import display, Markdown

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/'shared'/'implementation.ipynb').exists(): return p
    if (here/'part 1'/'shared'/'implementation.ipynb').exists(): return here/'part 1'
    raise FileNotFoundError('Run with the working directory inside part 1.')
ROOT=find_root()
os.environ.setdefault('HF_HOME', str(ROOT/'.cache'/'huggingface'))
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4,os.cpu_count() or 1))
PAD,BOS,EOS,SEP=0,1,2,3
LANGS=('vi','ja')
DATASET='belumind/en-vi-ja-curated-500k-triplets'

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def atomic_torch(obj,path):
    path=Path(path); temp=path.with_suffix(path.suffix+'.tmp'); torch.save(obj,temp); temp.replace(path)
def amp():
    return torch.autocast('cuda',dtype=torch.float16) if DEVICE.type=='cuda' else contextlib.nullcontext()

@dataclass
class Config:
    variant:int=1
    vocab:int=16000
    d:int=512
    heads:int=8
    layers:int=6
    hidden:int=2048
    context:int=256
    dropout:float=0.1
    seed:int=42
    token_budget:int=60_000_000
    batch_tokens:int=8192
    batch_examples:int=128
    lr:float=0.0006
    weight_decay:float=0.1
    balance_coef:float=0.01
    z_coef:float=0.001
    eval_interval:int=5_000_000
    val_examples:int=2048
    max_new_tokens:int=160
    profile:str='full'

NAMES={1:'Dense two-layer MLP',2:'4 experts; top 1; equal total parameters',
       3:'4 experts; top 2; equal total parameters',
       4:'1 always-active shared + 3 routed; top 1; equal total parameters',
       5:'4 experts; top 2; equal active parameters'}


## Parameter accounting and routing
The dense FFN has two bias-free matrices, so its budget is `2*d*H`. Models 2–4 allocate `4*d` trainable routing parameters and distribute `H-2` hidden units among four experts. At `H=2048`, widths are 512, 512, 511, 511. Model 4 spends `3*d` on routed logits and `d` on a sigmoid gate for the always-active shared expert; the shared expert is evaluated for every non-padding token. The sigmoid weights its output but never selects it out.

Model 5 uses four width-1023 experts: two active experts plus the router contain exactly `2*d*2048` parameters per token. There are no dummy parameters. Active parameter counts include router parameters and the complete common backbone (including tied embedding/output weights), and exclude inactive expert matrices. Models 2–4 have tiny route-dependent active-count differences due to integer widths; minimum and maximum are reported.

Top-1 uses the selected full-softmax probability, preserving task-loss gradients into the router. Top-2 renormalizes selected probabilities. Routing is sparse, does not drop tokens, and excludes padding from dispatch and auxiliary losses. Balance loss uses detached assignment frequencies; z-loss penalizes router log-partition magnitude. The auxiliary terms are excluded from perplexity.


In [ ]:
class MLP(nn.Module):
    def __init__(self,d,h):
        super().__init__(); self.up=nn.Linear(d,h,bias=False); self.down=nn.Linear(h,d,bias=False)
    def forward(self,x): return self.down(F.gelu(self.up(x)))

class FFN(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.variant=cfg.variant; self.capture=False; self.last=None
        if cfg.variant==1:
            self.dense=MLP(cfg.d,cfg.hidden); self.widths=[cfg.hidden]; self.topk=1
        else:
            self.topk=1 if cfg.variant in (2,4) else 2
            if cfg.variant==5:
                assert cfg.hidden%2==0
                self.widths=[cfg.hidden//2-1]*4
            else:
                q,r=divmod(cfg.hidden-2,4); self.widths=[q+(i<r) for i in range(4)]
            self.experts=nn.ModuleList([MLP(cfg.d,h) for h in self.widths])
            self.router=nn.Linear(cfg.d,3 if cfg.variant==4 else 4,bias=False)
            if cfg.variant==4: self.shared_gate=nn.Linear(cfg.d,1,bias=False)
    def forward(self,x,valid):
        if self.variant==1: return self.dense(x),x.new_zeros(()),x.new_zeros(())
        shape=x.shape; flat=x.reshape(-1,shape[-1]); positions=valid.reshape(-1).nonzero().squeeze(1)
        tokens=flat[positions]
        logits=F.linear(tokens.float(),self.router.weight.float())
        probs=logits.softmax(-1); weights,indices=probs.topk(self.topk,dim=-1)
        if self.topk>1: weights=weights/weights.sum(-1,keepdim=True)
        mixed=torch.zeros_like(tokens)
        offset=1 if self.variant==4 else 0
        for e,expert in enumerate(self.experts[offset:]):
            row,slot=torch.where(indices==e)
            if row.numel():
                values=expert(tokens[row])*weights[row,slot,None].to(tokens.dtype)
                mixed.index_add_(0,row,values.to(mixed.dtype))
        if self.variant==4:
            gate=torch.sigmoid(F.linear(tokens.float(),self.shared_gate.weight.float())).to(tokens.dtype)
            mixed=mixed+gate*self.experts[0](tokens)
        out=torch.zeros_like(flat); out.index_copy_(0,positions,mixed)
        assignments=F.one_hot(indices,num_classes=probs.shape[-1]).float().sum(1)/self.topk
        balance=probs.shape[-1]*(assignments.detach().mean(0)*probs.mean(0)).sum()
        zloss=logits.logsumexp(-1).square().mean()
        if self.capture:
            self.last=(positions.detach(),indices.detach()+offset,weights.detach())
        return out.reshape(shape),balance,zloss

class Attention(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.heads=cfg.heads; self.hd=cfg.d//cfg.heads; self.dropout=cfg.dropout
        assert self.hd%2==0
        self.qkv=nn.Linear(cfg.d,3*cfg.d,bias=False); self.out=nn.Linear(cfg.d,cfg.d,bias=False)
        angles=torch.outer(torch.arange(cfg.context),10000.0**(-torch.arange(0,self.hd,2).float()/self.hd))
        self.register_buffer('cos',angles.cos(),persistent=False); self.register_buffer('sin',angles.sin(),persistent=False)
    def rotate(self,x,positions):
        co=self.cos[positions][:,None].to(x.dtype); si=self.sin[positions][:,None].to(x.dtype)
        a,b=x[...,::2],x[...,1::2]
        return torch.stack((a*co-b*si,a*si+b*co),-1).flatten(-2)
    def forward(self,x,valid,positions,cache=None):
        b,t,d=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.hd).permute(2,0,3,1,4).unbind(0)
        q,k=self.rotate(q,positions),self.rotate(k,positions)
        if cache is not None:
            k=torch.cat((cache[0],k),dim=-2); v=torch.cat((cache[1],v),dim=-2)
        # Training uses right padding: no real token can see future padding.
        if cache is None:
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.)
        else:
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=valid[:,None,None,:],dropout_p=0.)
        return self.out(y.transpose(1,2).reshape(b,t,d)),(k,v)

class Block(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.n1=nn.LayerNorm(cfg.d); self.n2=nn.LayerNorm(cfg.d)
        self.attn=Attention(cfg); self.ffn=FFN(cfg); self.drop=nn.Dropout(cfg.dropout)
    def forward(self,x,valid,positions,cache=None):
        a,new_cache=self.attn(self.n1(x),valid,positions,cache); x=x+self.drop(a)
        local_valid=valid[:,-x.shape[1]:]
        y,bal,z=self.ffn(self.n2(x),local_valid)
        return x+self.drop(y),bal,z,new_cache

class Transformer(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.cfg=cfg
        self.embedding=nn.Embedding(cfg.vocab,cfg.d,padding_idx=PAD)
        self.blocks=nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.d); self.drop=nn.Dropout(cfg.dropout)
        self.apply(self.init)
        # Common parameters have exactly the same initialization in every variant.
        for name,p in self.named_parameters():
            if '.ffn.' not in name and p.ndim>=2:
                gen=torch.Generator().manual_seed(cfg.seed+int(hashlib.sha256(name.encode()).hexdigest()[:8],16))
                with torch.no_grad(): p.normal_(0,0.02,generator=gen)
        with torch.no_grad(): self.embedding.weight[PAD].zero_()
    @staticmethod
    def init(m):
        if isinstance(m,(nn.Linear,nn.Embedding)): nn.init.normal_(m.weight,std=0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self,ids,labels=None,valid=None,positions=None,cache=None):
        if valid is None: valid=ids.ne(PAD)
        if positions is None: positions=torch.arange(ids.shape[1],device=ids.device)[None].expand_as(ids)
        x=self.drop(self.embedding(ids)); bal=x.new_zeros(()); z=x.new_zeros(()); next_cache=[]
        for i,block in enumerate(self.blocks):
            x,b,zz,kv=block(x,valid,positions,None if cache is None else cache[i]); bal=bal+b; z=z+zz; next_cache.append(kv)
        hidden=self.norm(x)
        # Materialize vocabulary logits only at supervised positions during training/evaluation.
        if labels is not None:
            mask=labels.ne(-100)
            logits=F.linear(hidden[mask],self.embedding.weight)
            losses=F.cross_entropy(logits.float(),labels[mask],reduction='none')
            return losses,bal/self.cfg.layers,z/self.cfg.layers,mask
        return F.linear(hidden,self.embedding.weight),next_cache

def parameter_counts(cfg):
    with torch.device('meta'): model=Transformer(cfg)
    total=sum(p.numel() for p in model.parameters())
    expert=sum(p.numel() for n,p in model.named_parameters() if '.ffn.' in n and ('experts' in n or 'dense' in n))
    common=total-expert
    widths=model.blocks[0].ffn.widths
    if cfg.variant==1: low=high=2*cfg.d*widths[0]
    elif cfg.variant==4:
        low=2*cfg.d*(widths[0]+min(widths[1:])); high=2*cfg.d*(widths[0]+max(widths[1:]))
    else:
        k=model.blocks[0].ffn.topk; low=2*cfg.d*sum(sorted(widths)[:k]); high=2*cfg.d*sum(sorted(widths)[-k:])
    return {'model':cfg.variant,'total':total,'active_min':common+cfg.layers*low,'active_max':common+cfg.layers*high,'expert_widths':widths}


## Data and reproducible exposure
Official splits are retained. Normalized exact English-target overlap is audited; training rows with an English target appearing in either holdout split are removed, and validation rows sharing a test target are removed before model selection. Within-training exact triplet duplicates are removed. This prevents the same English sentence from entering both training and evaluation but cannot detect paraphrases or all noisy alignment.

The byte-level BPE tokenizer is trained on 50,000 cleaned training triplets only, preserving Vietnamese and Japanese Unicode scripts. Each triplet yields both source directions. Overlength examples and literal reserved-token collisions are counted and dropped without silently truncating references. Test BLEU uses a fixed seed-2026 sample of 1,000 retained test triplets, paired across languages and models. Perplexity covers all retained test pairs.

Length-bucketed training batches and their order depend only on the seed and shared dataset. A precomputed sequence of whole batches stays within the requested token budget; the exact realized budget is printed and is identical for every architecture. Both non-padding input tokens and supervised English/EOS tokens are counted. Padding and auxiliary losses never enter perplexity.


In [ ]:
REVISION='849990daee76e0f9e2eb9965e30e34bc1909a93d'
def canonical(text): return unicodedata.normalize('NFC',text).strip()

def prepare_data():
    from datasets import load_dataset
    base=ROOT/'shared'/'data'; base.mkdir(exist_ok=True)
    manifest_path=base/'manifest.json'
    if manifest_path.exists(): return json.loads(manifest_path.read_text())
    raw=load_dataset(DATASET,revision=REVISION)
    test_targets={canonical(t) for t in raw['test']['en']}
    val_targets={canonical(t) for t in raw['validation']['en']}
    held=test_targets|val_targets
    clean={}; audit={}; seen=set()
    for split in ('train','validation','test'):
        indices=[]; removed_overlap=0; removed_duplicate=0
        for i,row in enumerate(raw[split]):
            target=canonical(row['en'])
            if (split=='train' and target in held) or (split=='validation' and target in test_targets):
                removed_overlap+=1; continue
            key=tuple(canonical(row[k]) for k in ('vi','ja','en'))
            if split=='train' and key in seen: removed_duplicate+=1; continue
            if split=='train': seen.add(key)
            indices.append(i)
        clean[split]=raw[split].select(indices)
        audit[split]={'original_rows':len(raw[split]),'rows_after_overlap_dedup':len(indices),'removed_target_overlap':removed_overlap,'removed_train_triplet_duplicate':removed_duplicate}
    tok=Tokenizer(models.BPE()); tok.normalizer=normalizers.NFC()
    tok.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False); tok.decoder=decoders.ByteLevel()
    trainer=trainers.BpeTrainer(vocab_size=16000,min_frequency=2,special_tokens=['[PAD]','[BOS]','[EOS]','[SEP]'],initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),show_progress=False)
    sample=clean['train'].shuffle(seed=42).select(range(min(50000,len(clean['train']))))
    def corpus():
        for start in range(0,len(sample),1000):
            batch=sample[start:start+1000]
            yield [canonical(t) for lang in ('vi','ja','en') for t in batch[lang]]
    tok.train_from_iterator(corpus(),trainer=trainer); tok.save(str(ROOT/'shared'/'tokenizer_clean.json'))
    assert tok.get_vocab_size()==16000
    for i,s in enumerate(['[PAD]','[BOS]','[EOS]','[SEP]']): assert tok.token_to_id(s)==i
    token_stats={}
    for lang in ('vi','ja','en'):
        texts=[canonical(t) for t in clean['validation'][:1000][lang]]
        enc=tok.encode_batch(texts,add_special_tokens=False)
        assert all(tok.decode(e.ids)==t for e,t in zip(enc,texts))
        token_stats[lang]={'mean_tokens':float(np.mean([len(e.ids) for e in enc])),'roundtrip_checked':len(texts)}
    for split,ds in clean.items():
        seqs=[]; seps=[]; langs=[]; rows=[]; raw_ids=[]; dropped=Counter(); retained_rows=[]
        for start in range(0,len(ds),2000):
            batch=ds[start:start+2000]
            enc={lang:tok.encode_batch([canonical(t) for t in batch[lang]],add_special_tokens=False) for lang in ('vi','ja','en')}
            for j in range(len(batch['en'])):
                pair=[]
                for lang_id,lang in enumerate(LANGS):
                    src,tgt=enc[lang][j].ids,enc['en'][j].ids
                    if set(src+tgt)&{PAD,BOS,EOS,SEP}: dropped['reserved_token']+=1; continue
                    ids=[BOS,*src,SEP,*tgt,EOS]
                    if len(ids)-1>256: dropped['overlength_'+lang]+=1; continue
                    if not src or not tgt: dropped['empty']+=1; continue
                    pair.append((ids,len(src)+1,lang_id))
                # Paired filtering guarantees identical reference rows for both directions.
                if len(pair)!=2: dropped['triplets_removed']+=1; continue
                row_id=len(retained_rows)
                retained_rows.append({k:canonical(batch[k][j]) for k in ('en','vi','ja')} | {'quality_score':float(batch['quality_score'][j]),'source':batch['source'][j]})
                for ids,sep,lang_id in pair:
                    seqs.append(np.asarray(ids,dtype=np.int32)); seps.append(sep); langs.append(lang_id); rows.append(row_id)
        folder=base/split; folder.mkdir(exist_ok=True)
        lengths=np.array([len(s) for s in seqs],dtype=np.int32)
        offsets=np.r_[0,lengths.cumsum(dtype=np.int64)]
        for name,arr in {'flat':np.concatenate(seqs),'offsets':offsets,'seps':np.array(seps,dtype=np.int32),'langs':np.array(langs,dtype=np.int8),'rows':np.array(rows,dtype=np.int32)}.items(): np.save(folder/(name+'.npy'),arr)
        (folder/'raw.json').write_text(json.dumps(retained_rows,ensure_ascii=False))
        audit[split].update({'retained_triplets':len(retained_rows),'pairs':len(seqs),'input_tokens':int((lengths-1).sum()),'target_tokens':int((lengths-1-np.array(seps)).sum()),'drops':dict(dropped)})
        print(split,audit[split],flush=True)
    manifest={'dataset':DATASET,'revision':REVISION,'tokenizer_sha256':sha(ROOT/'shared'/'tokenizer_clean.json'),'audit':audit,'tokenizer_stats':token_stats,'created_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
    manifest_path.write_text(json.dumps(manifest,indent=2)); return manifest

class PairStore:
    def __init__(self,split):
        self.folder=ROOT/'shared'/'data'/split
        for name in ('flat','offsets','seps','langs','rows'): setattr(self,name,np.load(self.folder/(name+'.npy'),mmap_mode='r'))
        self.lengths=np.diff(self.offsets)-1
    def __len__(self): return len(self.seps)
    def get(self,i): return self.flat[self.offsets[i]:self.offsets[i+1]],int(self.seps[i]),int(self.langs[i])

def collate(ds,indices):
    length=int(max(ds.lengths[indices])); x=torch.zeros((len(indices),length),dtype=torch.long); y=torch.full_like(x,-100)
    lang=torch.tensor(np.asarray(ds.langs[indices]).copy(),dtype=torch.long); seps=torch.tensor(np.asarray(ds.seps[indices]).copy(),dtype=torch.long)
    for row,i in enumerate(indices):
        ids,sep,_=ds.get(i); ids=torch.from_numpy(np.array(ids,dtype=np.int64)); n=len(ids)-1
        x[row,:n]=ids[:-1]; y[row,sep:n]=ids[sep+1:]
    return x.to(DEVICE),y.to(DEVICE),lang.to(DEVICE),seps.to(DEVICE)

def epoch_batches(ds,cfg,epoch=0,indices=None,shuffle=True):
    rng=np.random.default_rng(cfg.seed+epoch)
    order=np.arange(len(ds)) if indices is None else np.asarray(indices).copy()
    if shuffle: rng.shuffle(order)
    batches=[]
    for start in range(0,len(order),4096):
        pool=order[start:start+4096]; pool=pool[np.argsort(ds.lengths[pool],kind='stable')]
        batch=[]; longest=0
        for i in pool:
            size=int(ds.lengths[i]); new_long=max(longest,size)
            if batch and ((len(batch)+1)*new_long>cfg.batch_tokens or len(batch)>=cfg.batch_examples):
                batches.append(np.array(batch)); batch=[]; longest=0
            batch.append(int(i)); longest=max(longest,size)
        if batch: batches.append(np.array(batch))
    if shuffle: rng.shuffle(batches)
    return batches

def schedule(ds,cfg,budget):
    result=[]; count=0; epoch=0
    while True:
        batches=epoch_batches(ds,cfg,epoch)
        assert batches
        for batch in batches:
            n=int(ds.lengths[batch].sum())
            if count+n>budget:
                assert result, 'Budget must fit at least one whole batch.'
                return result,count
            result.append(batch); count+=n
        epoch+=1

def schedule_digest(batches):
    h=hashlib.sha256()
    for b in batches: h.update(np.asarray(b,dtype='<i8').tobytes()); h.update(b'|')
    return h.hexdigest()

@torch.inference_mode()
def evaluate_nll(model,ds,cfg,indices=None):
    model.eval(); nll=np.zeros(2); counts=np.zeros(2,dtype=np.int64); correct=np.zeros(2,dtype=np.int64)
    for batch in epoch_batches(ds,cfg,indices=indices,shuffle=False):
        x,y,lang,seps=collate(ds,batch)
        with amp(): losses,_,_,mask=model(x,labels=y)
        groups=lang[:,None].expand_as(y)[mask]
        for g in range(2):
            selected=groups==g; nll[g]+=float(losses[selected].double().sum()); counts[g]+=int(selected.sum())
    assert (counts>0).all()
    result={'nll':float(nll.sum()/counts.sum()),'ppl':math.exp(float(nll.sum()/counts.sum())),'target_tokens':int(counts.sum())}
    for g,lang in enumerate(LANGS): result.update({f'nll_{lang}':float(nll[g]/counts[g]),f'ppl_{lang}':math.exp(float(nll[g]/counts[g]))})
    return result


In [ ]:
def training_run(cfg,train,val,out,budget,resume=True):
    out=Path(out); out.mkdir(exist_ok=True,parents=True)
    batches,realized=schedule(train,cfg,budget); digest=schedule_digest(batches)
    seed_all(cfg.seed); model=Transformer(cfg).to(DEVICE)
    groups=[{'params':[p for p in model.parameters() if p.ndim>=2],'weight_decay':cfg.weight_decay},
            {'params':[p for p in model.parameters() if p.ndim<2],'weight_decay':0.}]
    optimizer=torch.optim.AdamW(groups,lr=cfg.lr,betas=(.9,.95),eps=1e-8)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda',init_scale=128.)
    resume_path=out/'resume.pt'; history=[]; done=0; target_done=0; start_step=0; elapsed_before=0.
    if resume and resume_path.exists():
        saved=torch.load(resume_path,map_location='cpu',weights_only=False)
        assert saved['config']==asdict(cfg) and saved['schedule_digest']==digest
        model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer']); scaler.load_state_dict(saved['scaler'])
        history=saved['history']; done=saved['input_tokens']; target_done=saved['target_tokens']; start_step=saved['step']; elapsed_before=saved['seconds']
        torch.set_rng_state(saved['torch_rng'])
        if DEVICE.type=='cuda': torch.cuda.set_rng_state_all(saved['cuda_rng'])
        del saved
    val_idx=np.sort(np.random.default_rng(109).choice(len(val),min(cfg.val_examples,len(val)),replace=False))
    started=time.monotonic(); running_nll=0.; running_targets=0; next_eval=(done//cfg.eval_interval+1)*cfg.eval_interval
    def save_resume(step):
        atomic_torch({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),'config':asdict(cfg),
                      'schedule_digest':digest,'history':history,'input_tokens':done,'target_tokens':target_done,'step':step,
                      'seconds':elapsed_before+time.monotonic()-started,'torch_rng':torch.get_rng_state(),
                      'cuda_rng':torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else []},resume_path)
    for step in range(start_step,len(batches)):
        model.train(); indices=batches[step]; x,y,lang,seps=collate(train,indices)
        n=int(train.lengths[indices].sum()); nt=int(y.ne(-100).sum())
        progress=done/max(1,realized); warm=.03
        factor=max(.01,progress/warm) if progress<warm else .1+.9*.5*(1+math.cos(math.pi*(progress-warm)/(1-warm)))
        for group in optimizer.param_groups: group['lr']=cfg.lr*factor
        optimizer.zero_grad(set_to_none=True)
        with amp():
            losses,bal,z,_=model(x,labels=y)
            loss=losses.mean()+cfg.balance_coef*bal+cfg.z_coef*z
        if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite loss at step {step}')
        scale_before=scaler.get_scale(); scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        skipped=scaler.get_scale()<scale_before
        if skipped: raise FloatingPointError('AMP skipped an update; reduce loss scale and restart equally for all variants.')
        done+=n; target_done+=nt; running_nll+=float(losses.detach().sum()); running_targets+=nt
        if (step+1)%100==0 or step+1==len(batches):
            record={'kind':'train','step':step+1,'input_tokens':done,'target_tokens':target_done,'nll':running_nll/running_targets,
                    'balance':float(bal.detach()),'router_z':float(z.detach()),'grad_norm':float(grad_norm),
                    'lr':optimizer.param_groups[0]['lr'],'seconds':elapsed_before+time.monotonic()-started}
            history.append(record); running_nll=0.; running_targets=0
            print(f"model {cfg.variant} | {done:,}/{realized:,} input tokens | nll {record['nll']:.4f} | {record['seconds']:.0f}s",flush=True)
        if done>=next_eval or step+1==len(batches):
            score=evaluate_nll(model,val,cfg,val_idx); history.append({'kind':'validation','input_tokens':done,**score})
            print('validation',score,flush=True); next_eval+=cfg.eval_interval; save_resume(step+1)
    assert done==realized
    summary={'requested_budget':budget,'input_tokens':done,'target_tokens':target_done,'steps':len(batches),
             'schedule_digest':digest,'seconds':elapsed_before+time.monotonic()-started,'validation_selection':history[-1]}
    atomic_torch({'model':model.state_dict(),'config':asdict(cfg),'summary':summary,'history':history,
                  'tokenizer_sha256':sha(ROOT/'shared'/'tokenizer_clean.json')},out/'checkpoint.pt')
    if budget!=cfg.token_budget and resume_path.exists(): resume_path.unlink()
    return model,summary,history

@torch.inference_mode()
def greedy_translations(model,tok,texts,cfg,batch_size=32):
    model.eval(); results=[]; capped=[]
    encoded=[e.ids for e in tok.encode_batch([canonical(t) for t in texts],add_special_tokens=False)]
    for start in range(0,len(encoded),batch_size):
        sources=encoded[start:start+batch_size]; prompts=[[BOS,*s,SEP] for s in sources]
        assert all(len(p)<cfg.context for p in prompts),'Source exceeds context; do not silently truncate it.'
        lens=torch.tensor([len(p) for p in prompts],device=DEVICE); b=len(prompts); width=int(lens.max())
        x=torch.zeros((b,width),dtype=torch.long,device=DEVICE)
        for i,p in enumerate(prompts): x[i,:len(p)]=torch.tensor(p,device=DEVICE)
        valid=x.ne(PAD)
        with amp(): logits,cache=model(x)
        next_logits=logits[torch.arange(b,device=DEVICE),lens-1]; alive=torch.ones(b,dtype=torch.bool,device=DEVICE)
        generated=[[] for _ in prompts]; hit_cap=[False]*b
        for t in range(cfg.max_new_tokens):
            next_logits[:,[PAD,BOS,SEP]]=-float('inf'); token=next_logits.argmax(-1)
            for i in range(b):
                if alive[i]:
                    if int(token[i])==EOS: alive[i]=False
                    else: generated[i].append(int(token[i]))
                if alive[i] and (int(lens[i])+t>=cfg.context-1 or t==cfg.max_new_tokens-1): alive[i]=False; hit_cap[i]=True
            if not bool(alive.any()): break
            token=torch.where(alive,token,torch.full_like(token,EOS)); valid=torch.cat([valid,torch.ones((b,1),device=DEVICE,dtype=torch.bool)],1)
            positions=(lens+t).clamp(max=cfg.context-1)[:,None]
            with amp(): logits,cache=model(token[:,None],valid=valid,positions=positions,cache=cache)
            next_logits=logits[:,0]
        results.extend(tok.decode(seq) for seq in generated); capped.extend(hit_cap)
    return results,capped

@torch.inference_mode()
def evaluate_bleu(model,tok,cfg,split='test',nrows=1000,seed=2026,bootstrap=300):
    rows=json.loads((ROOT/'shared'/'data'/split/'raw.json').read_text())
    chosen=np.sort(np.random.default_rng(seed).choice(len(rows),min(nrows,len(rows)),replace=False))
    selected=[rows[i] for i in chosen]; records=[]; result={'triplets':len(selected),'sample_seed':seed,'sample_rows_sha256':hashlib.sha256(chosen.astype('<i8').tobytes()).hexdigest()}
    metric=sacrebleu.metrics.BLEU(); started=time.monotonic()
    for lang in LANGS:
        hyps,caps=greedy_translations(model,tok,[r[lang] for r in selected],cfg)
        refs=[r['en'] for r in selected]; result['bleu_'+lang]=metric.corpus_score(hyps,[refs]).score
        result['generation_cap_'+lang]=sum(caps)
        records.extend({'row':int(i),'language':lang,'source':r[lang],'reference':ref,'hypothesis':hyp,'capped':cap,'quality_score':r['quality_score'],'corpus_source':r['source']} for i,r,ref,hyp,cap in zip(chosen,selected,refs,hyps,caps))
    result['bleu']=metric.corpus_score([r['hypothesis'] for r in records],[[r['reference'] for r in records]]).score
    result['signature']=str(metric.get_signature()); result['seconds']=time.monotonic()-started
    # Paired triplet resampling keeps vi/ja observations of the same target together.
    rng=np.random.default_rng(314); stats=[]; n=len(selected)
    for _ in range(bootstrap):
        ix=rng.integers(0,n,n); res=[records[int(i)+offset] for offset in (0,n) for i in ix]
        stats.append(metric.corpus_score([r['hypothesis'] for r in res],[[r['reference'] for r in res]]).score)
    if stats: result['bleu_ci95']=np.percentile(stats,[2.5,97.5]).tolist()
    return result,records


In [ ]:
@torch.inference_mode()
def routing_diagnostics(model,ds,cfg,max_pairs=1000):
    if cfg.variant==1: return {'applicable':False,'reason':'Dense MLP has no router.'}
    for block in model.blocks: block.ffn.capture=True
    counts=np.zeros((cfg.layers,2,2,4),dtype=np.int64); totals=np.zeros((2,2),dtype=np.int64)
    indices=np.sort(np.random.default_rng(909).choice(len(ds),min(max_pairs,len(ds)),replace=False))
    for batch in epoch_batches(ds,cfg,indices=indices,shuffle=False):
        x,y,lang,seps=collate(ds,batch)
        with amp(): model(x,labels=y)
        positions=torch.arange(x.shape[1],device=DEVICE)[None].expand_as(x)
        regions=[(positions>0)&(positions<seps[:,None])&x.ne(PAD),(positions>seps[:,None])&x.ne(PAD)]
        for g in range(2):
            for r,region in enumerate(regions): totals[g,r]+=int((region&(lang[:,None]==g)).sum())
        for layer,block in enumerate(model.blocks):
            pos,expert,weights=block.ffn.last
            flat_lang=lang[:,None].expand_as(x).reshape(-1)[pos]
            for g in range(2):
                for r,region in enumerate(regions):
                    mask=region.reshape(-1)[pos]&(flat_lang==g)
                    if mask.any(): counts[layer,g,r]+=torch.bincount(expert[mask].flatten(),minlength=4).cpu().numpy()
                    if cfg.variant==4: counts[layer,g,r,0]+=int(mask.sum())
    for block in model.blocks: block.ffn.capture=False; block.ffn.last=None
    rates=counts/np.maximum(totals[None,:,:,None],1)
    expected=1 if cfg.variant==2 else 2
    assert np.allclose(rates.sum(-1),expected)
    if cfg.variant==4: assert np.allclose(rates[...,0],1)
    diagnostics=[]
    for layer in range(cfg.layers):
        for g,lang in enumerate(LANGS):
            for r,region in enumerate(('source','English-prefix')):
                values=counts[layer,g,r,1:] if cfg.variant==4 else counts[layer,g,r]
                dist=values/max(1,values.sum()); entropy=float(-(dist[dist>0]*np.log(dist[dist>0])).sum()/np.log(len(dist)))
                diagnostics.append({'layer':layer+1,'language':lang,'region':region,'normalized_routing_entropy':entropy,'unused_experts':int((values==0).sum()),'max_assignment_share':float(dist.max())})
    return {'applicable':True,'selection_rates':rates.tolist(),'counts':counts.tolist(),'token_counts':totals.tolist(),'diagnostics':diagnostics,'sample_pairs':len(indices)}

def plot_model_results(history,routing):
    train=pd.DataFrame([r for r in history if r['kind']=='train']); val=pd.DataFrame([r for r in history if r['kind']=='validation'])
    fig,ax=plt.subplots(figsize=(8,4))
    if len(train): ax.plot(train.input_tokens/1e6,train.nll,label='Training target NLL')
    if len(val): ax.plot(val.input_tokens/1e6,val.nll,'o-',label='Validation target NLL')
    ax.set(xlabel='Non-padding training input tokens (millions)',ylabel='English target NLL'); ax.legend(); ax.grid(alpha=.2); plt.show()
    if routing.get('applicable'):
        rates=np.array(routing['selection_rates']); layers=rates.shape[0]
        fig,axes=plt.subplots(2,layers,figsize=(3*layers,6),squeeze=False,constrained_layout=True)
        for r,region in enumerate(('Source tokens','English-prefix tokens')):
            for layer in range(layers):
                ax=axes[r,layer]; im=ax.imshow(rates[layer,:,r],vmin=0,vmax=1,cmap='viridis',aspect='auto')
                ax.set(title=f'{region}, layer {layer+1}',xticks=range(4),xticklabels=[f'E{i}' for i in range(4)],yticks=[0,1],yticklabels=['vi','ja'])
                for g in range(2):
                    for e in range(4): ax.text(e,g,f'{rates[layer,g,r,e]:.2f}',ha='center',va='center',color='white' if rates[layer,g,r,e]<.5 else 'black',fontsize=8)
        fig.colorbar(im,ax=axes.ravel().tolist(),label='Fraction of region tokens selecting expert'); plt.show()
        display(pd.DataFrame(routing['diagnostics']))

def run_checks(variant):
    cfg=Config(variant=variant,vocab=96,d=32,heads=4,layers=2,hidden=128,context=32,dropout=0.,profile='checks')
    seed_all(7); model=Transformer(cfg).to(DEVICE).eval()
    x=torch.tensor([[BOS,9,10,SEP,11,12,EOS],[BOS,13,SEP,14,EOS,PAD,PAD]],device=DEVICE)
    labels=torch.tensor([[-100,-100,-100,11,12,EOS,-100],[-100,-100,14,EOS,-100,-100,-100]],device=DEVICE)
    with torch.no_grad():
        first,_=model(x); alt=x.clone(); alt[0,4]=17; changed,_=model(alt)
        assert torch.allclose(first[0,:4],changed[0,:4],atol=1e-6), 'Causal leakage'
        solo,_=model(x[1:2,:5]); assert torch.allclose(solo[0],first[1,:5],atol=1e-5),'Padding changes real logits'
        prefix=x[:1,:5]; full,_=model(prefix); pref,cache=model(prefix[:,:4])
        incremental,_=model(prefix[:,4:],cache=cache,valid=torch.ones((1,5),dtype=torch.bool,device=DEVICE),positions=torch.tensor([[4]],device=DEVICE))
        assert torch.allclose(full[:,-1],incremental[:,0],atol=1e-5), 'KV cache mismatch'
    losses,bal,z,mask=model(x,labels=labels)
    direct=F.cross_entropy(first[mask].float(),labels[mask],reduction='none')
    assert torch.allclose(losses,direct,atol=1e-6)
    (losses.mean()+.01*bal+.001*z).backward()
    if variant>1:
        for block in model.blocks:
            grad=block.ffn.router.weight.grad
            assert grad is not None and torch.isfinite(grad).all() and grad.abs().sum()>0,'Router has no learning signal'
        # Sparse dispatch equals an independently assembled dense expert mixture.
        f=model.blocks[0].ffn; tokens=torch.randn((2,5,cfg.d),device=DEVICE); valid=torch.ones((2,5),device=DEVICE,dtype=torch.bool)
        got,_,_=f(tokens,valid); flat=tokens.reshape(-1,cfg.d)
        probs=f.router(flat).softmax(-1); w,idx=probs.topk(f.topk,-1)
        if f.topk>1: w=w/w.sum(-1,keepdim=True)
        ref=torch.zeros_like(flat); offset=int(variant==4)
        for e,expert in enumerate(f.experts[offset:]):
            coefficient=(w*(idx==e)).sum(-1,keepdim=True); ref+=coefficient*expert(flat)
        if variant==4: ref+=f.shared_gate(flat).sigmoid()*f.experts[0](flat)
        assert torch.allclose(got.reshape_as(ref),ref,atol=1e-6),'Sparse dispatch mismatch'
    optimizer=torch.optim.AdamW(model.parameters(),lr=.006)
    initial=float(losses.mean().detach())
    for _ in range(35):
        optimizer.zero_grad(); l,b,z,_=model(x,labels=labels); (l.mean()+.01*b+.001*z).backward(); optimizer.step()
    final=float(model(x,labels=labels)[0].mean().detach()); assert final<initial*.6,'Tiny-batch overfit failed'
    stream=io.BytesIO(); torch.save(model.state_dict(),stream); stream.seek(0)
    copy=Transformer(cfg).to(DEVICE).eval(); copy.load_state_dict(torch.load(stream,map_location=DEVICE,weights_only=True))
    with torch.no_grad(): assert torch.equal(model(x)[0],copy(x)[0]),'Checkpoint round trip failed'
    result={'model':variant,'causal_attention':'PASS','padding':'PASS','target_loss':'PASS','KV_cache':'PASS','checkpoint':'PASS','tiny_overfit_initial':initial,'tiny_overfit_final':final,'sparse_dispatch':'PASS' if variant>1 else 'not applicable','router_gradient':'PASS' if variant>1 else 'not applicable'}
    del model,copy,optimizer; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    return result
